# Automated EDA Code Companion

This notebook shows a clean automated-style EDA workflow without installing external packages. It creates quick reports for shape, missing values, column types, numerical summaries, correlations, and target balance.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine


## Load a Dataset

EDA begins before modeling. The goal is to understand what the dataset contains and whether anything looks suspicious.


In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame

df.head()


## Basic Dataset Overview


In [ ]:
overview = pd.DataFrame({
    "value": [df.shape[0], df.shape[1], df.duplicated().sum(), df.isnull().sum().sum()]
}, index=["rows", "columns", "duplicate_rows", "total_missing_values"])

overview


## Column Types and Missing Values

This report quickly shows which columns are numeric and whether any column has missing data.


In [ ]:
column_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
    "unique_values": df.nunique()
})

column_report


## Numerical Summary

Summary statistics help detect scale differences, unusual ranges, and possible outliers.


In [ ]:
df.describe().T


## Target Balance

For classification datasets, target balance matters because imbalanced classes can make Accuracy misleading.


In [ ]:
target_counts = df["target"].value_counts().sort_index()
target_report = pd.DataFrame({
    "class_name": wine.target_names,
    "count": target_counts.values,
    "percent": (target_counts.values / len(df) * 100).round(2)
})

target_report


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(target_report["class_name"], target_report["count"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Class Balance")
plt.tight_layout()
plt.show()


## Correlation Report

Correlation helps identify relationships between numerical features. Very high correlations can indicate redundant features.


In [ ]:
correlation_with_target = df.corr(numeric_only=True)["target"].sort_values(ascending=False)
correlation_with_target


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(df.corr(numeric_only=True), cmap="coolwarm", aspect="auto")
plt.colorbar(label="correlation")
plt.xticks(range(len(df.columns)), df.columns, rotation=90)
plt.yticks(range(len(df.columns)), df.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## Simple Automated EDA Function

This function collects the most common EDA checks in one place. It is not a replacement for thinking, but it gives a fast first inspection.


In [ ]:
def quick_eda_report(dataframe, target_column=None):
    report = {
        "shape": dataframe.shape,
        "duplicate_rows": int(dataframe.duplicated().sum()),
        "total_missing_values": int(dataframe.isnull().sum().sum()),
        "column_report": pd.DataFrame({
            "dtype": dataframe.dtypes.astype(str),
            "missing_values": dataframe.isnull().sum(),
            "missing_percent": (dataframe.isnull().mean() * 100).round(2),
            "unique_values": dataframe.nunique()
        }),
        "numeric_summary": dataframe.describe().T
    }

    if target_column is not None:
        report["target_counts"] = dataframe[target_column].value_counts()

    return report

report = quick_eda_report(df, target_column="target")
report["shape"], report["duplicate_rows"], report["total_missing_values"]


In [ ]:
report["column_report"]
